# **Start**

In [1]:
# FIX K1: same notebook runs on Google Colab (Drive) or Kaggle (internet on).
# On Kaggle the working dir is /kaggle/working, so the original "./drive/MyDrive/..." paths
# below are reused unchanged; the dataset is copied there as train.csv / test.csv.
import os, glob, shutil
IN_KAGGLE = os.path.exists("/kaggle/working")
if IN_KAGGLE:
    _data_dir = "./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/data"
    os.makedirs(_data_dir, exist_ok=True)
    shutil.copy(glob.glob("/kaggle/input/**/d09_a.csv", recursive=True)[0], f"{_data_dir}/train.csv")  # water_quality_ewqi train
    shutil.copy(glob.glob("/kaggle/input/**/d09_b.csv", recursive=True)[0], f"{_data_dir}/test.csv")   # water_quality_ewqi test
    print("Kaggle: water_quality_ewqi data copied to", _data_dir)
else:
    from google.colab import drive
    drive.mount('/content/drive')


Kaggle: water_quality_ewqi data copied to ./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/data


In [2]:
# FIX K1: original Colab install block kept as-is (runs only on Colab).
# On Kaggle: install only the packages the Kaggle image lacks (it has numpy/sklearn/torch/xgboost/
# lightgbm/catboost/optuna/shap already), and do NOT restart the kernel with os._exit(0).
if IN_KAGGLE:
    !pip install -q optuna==4.6.0 optunahub==0.4.0 cmaes==0.12.0 xgboost==3.1.2 catboost==1.2.8 gpboost==1.6.1 ngboost==0.5.8 pgbm==2.2.0 pytorch-tabnet2==4.5.3 interpret==0.7.4 lime==0.2.0.1 bayesian-optimization==3.2.0 properscoring==0.1 XlsxWriter==3.2.9
else:
    import os
    os.environ["PIP_CONSTRAINT"] = "/tmp/numpy_constraint.txt"
    !echo "numpy==1.26.4" > /tmp/numpy_constraint.txt

    !pip uninstall -y torch torchaudio torchvision \
        torchao torchcodec torchdata torchtune torchsummary -q 2>/dev/null

    !pip uninstall -y tensorflow tensorflow-text tensorflow-hub tf-keras \
        tensorflow_decision_forests tensorflow-probability \
        tensorflow-Datasets tensorflow-metadata -q 2>/dev/null

    !pip uninstall -y numpy scikit-learn shap xgboost lightgbm dask \
        seaborn plotly openpyxl Cython catboost interpret lime -q 2>/dev/null

    !pip install numpy==1.26.4 -q
    !pip install scikit-learn==1.6.1 -q
    !pip install torch==2.9.0 -q

    !pip install lightgbm==4.6.0 -q
    !pip install xgboost==3.1.2 -q
    !pip install catboost==1.2.8 -q
    !pip install gpboost==1.6.1 -q
    !pip install ngboost==0.5.8 -q
    !pip install pgbm==2.2.0 -q
    !pip install pytorch-tabnet2==4.5.3 -q

    !pip install bayesian-optimization==3.2.0 -q
    !pip install optuna==4.6.0 -q
    !pip install optunahub==0.4.0 -q
    !pip install cmaes==0.12.0 -q

    !pip install shap==0.44.0 -q
    !pip install lime==0.2.0.1 -q
    !pip install interpret==0.7.4 -q

    !pip install mapie==0.6.0 -q
    !pip install puncc==0.8.0 -q
    !pip install skorch==1.3.1 -q
    !pip install properscoring==0.1 -q

    !pip install dask[dataframe]==2025.12.0 -q
    !pip install cython==3.0.12 -q
    !pip install seaborn==0.13.2 -q
    !pip install plotly==5.24.1 -q
    !pip install kaleido==1.2.0 -q
    !pip install openpyxl==3.1.5 -q
    !pip install XlsxWriter==3.2.9 -q
    !pip install cp==2020.12.3 -q

    !pip install numpy==1.26.4 --force-reinstall --no-deps -q

    os._exit(0)


  Installing build dependencies ... done


  Getting requirements to build wheel ... done


  Preparing metadata (pyproject.toml) ... done


  Installing build dependencies ... done


  Getting requirements to build wheel ... done


  Preparing metadata (pyproject.toml) ... done


  Installing build dependencies ... done


  Getting requirements to build wheel ... done


  Preparing metadata (pyproject.toml) ... done


# **Imports**

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ngboost
import gpboost
from scipy.stats import randint
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold  # FIX 1
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.svm import SVR
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from gpboost import GPBoostRegressor
from ngboost import NGBRegressor
import optuna
import optunahub
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingGridSearchCV
from interpret import show
from interpret.blackbox import LimeTabular, ShapKernel
from optuna.samplers import RandomSampler
import random
import time
from ngboost.distns import Normal
from ngboost.scores import LogScore
from scipy.stats import norm
from interpret import set_visualize_provider
from interpret.provider import InlineProvider
from interpret.glassbox import ExplainableBoostingRegressor
from interpret import show
import plotly.express as px
from io import BytesIO
from openpyxl import Workbook, load_workbook
import os
from openpyxl.drawing.image import Image as openpyxlImage
import warnings
import xlsxwriter
from openpyxl.drawing.image import Image
from pgbm.sklearn import HistGradientBoostingRegressor
import torch
from pgbm.torch import PGBM
import plotly.graph_objects as go
warnings.filterwarnings('ignore')
import pickle
import json
from pytorch_tabnet import TabNetRegressor

[1/2] c++ -MMD -MF splitgain_cpu.o.d -DTORCH_EXTENSION_NAME=split_decision -DTORCH_API_INCLUDE_EXTENSION_H -isystem /usr/local/lib/python3.13/dist-packages/torch/include -isystem /usr/local/lib/python3.13/dist-packages/torch/include/torch/csrc/api/include -isystem /usr/include/python3.13 -fPIC -std=c++17 -c /usr/local/lib/python3.13/dist-packages/pgbm/torch/splitgain_cpu.cpp -o splitgain_cpu.o 


[2/2] c++ splitgain_cpu.o -shared -L/usr/local/lib/python3.13/dist-packages/torch/lib -lc10 -ltorch_cpu -ltorch -ltorch_python -o split_decision.so


In [4]:
# Go to find & replace button and replace (water_quality_ewqi_uncertainty_analysis) with your folder name. Rename your train and test Dataset as train.csv and test.csv.
# Modify the names of the feature in the below cell.
# Replace (str) with actual data label name.

In [5]:
feature_names = ['pH', 'TDS', 'EC', 'TH', 'ca', 'mg', 'CO3', 'HCO3', 'NA', 'K', 'CHLORIDE', 'SULPHATE', 'NITRATE', 'FLUORIDE']
TARGET_LABEL = 'Pre-monsoon EWQI'  # FIX B12: target name used in plot labels (was 'str', the concrete dataset's target, on every dataset)

In [6]:
train_data_path = "./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/data/train.csv"
test_data_path = "./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/data/test.csv"
train_data = pd.read_csv(train_data_path)
test_data = pd.read_csv(test_data_path)
print("Training data loaded successfully.")
print("Test data loaded successfully.")

Training data loaded successfully.
Test data loaded successfully.


In [7]:
print("\nShape of training data:", train_data.shape)
print("First 5 rows of training data:\n", train_data.head(5))
print("\nShape of test data:", test_data.shape)
print("First 5 rows of test data:\n", test_data.head(5))


Shape of training data: (47, 15)
First 5 rows of training data:
      pH   TDS    EC    TH   ca    mg  CO3  HCO3   NA    K  CHLORIDE  SULPHATE  \
0  7.90   552   865   452  272   180  0.2   4.5  260   44       375       681   
1  7.52  1985  2256  1054  529   525  0.0  15.0  272   47       728       324   
2  7.65  1865  2032   636  371   265  0.0   9.0  373   25       713       503   
3  7.25  4855  7856  2050  795  1255  0.0  12.5  335  106        81       676   
4  7.96  2855  3696   865  310   555  0.3   4.6  184  106       698       324   

   NITRATE  FLUORIDE  Pre monsoon EWQI   
0       60      0.40              70.52  
1       10      0.65             138.78  
2       54      0.85             113.07  
3       17      0.69             266.32  
4       25      1.20             158.05  

Shape of test data: (47, 15)
First 5 rows of test data:
      pH   TDS    EC    TH     ca      mg  CO3  HCO3   NA    K  CHLORIDE  \
0  8.38   625   975   320  120.0   200.0  0.0   5.8  280   35 

In [8]:
X_train = train_data.iloc[:, :-1]
y_train = train_data.iloc[:, -1]
X_test = test_data.iloc[:, :-1]
y_test = test_data.iloc[:, -1]
# FIX D4 / D5 (approved 2026-11-22): settings that follow the test-set size
N_BINS = max(2, min(10, len(y_test) // 10))  # D4: bins for binned metrics (was 10 on every dataset; pile: 2-3 points per bin)
COP_T_BURNIN = min(50, len(y_test) // 2)  # D5: COP burn-in (was 50, so with < 50 test points COP's correction never started)
x_test= X_test
print("\nShape of X_train:", X_train.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of X_test:", X_test.shape)
print("Shape of y_test:", y_test.shape)


Shape of X_train: (47, 14)
Shape of y_train: (47,)
Shape of X_test: (47, 14)
Shape of y_test: (47,)


In [9]:
# Apply z-score normalization
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Print the first five rows of the normalized data
print("\nFirst five rows of normalized X_train:")
print(X_train[:5])

print("\nFirst five rows of normalized X_test:")
print(X_test[:5])


First five rows of normalized X_train:
[[ 0.3797749  -1.15230942 -0.94309903 -0.47923131 -0.09229732 -0.56852482
  -0.03922784 -1.18768739 -0.02713192 -0.87884263 -0.2770724   1.15873613
   1.41276153 -1.25316406]
 [-0.78765707 -0.2953837  -0.52059702  0.45803078  1.31357822  0.10564733
  -0.50015494  1.61405127  0.07159325 -0.77136925  1.29937074 -0.45273785
  -1.61366471 -0.53733141]
 [-0.38827245 -0.36714301 -0.58863473 -0.19275917  0.44926563 -0.40242444
  -0.50015494  0.01305775  0.90253007 -1.55950739  1.23238307  0.35525611
   1.04959038  0.03533472]
 [-1.61714822  1.42085972  1.18034563  2.00871688  2.76868675  1.53215652
  -0.50015494  0.94697063  0.58990037  1.34227396 -1.59003071  1.13616647
  -1.18996504 -0.42279818]
 [ 0.56410627  0.22487128 -0.08321177  0.16377408  0.11557533  0.16427099
   0.19123571 -1.16100417 -0.6523913   1.34227396  1.1653954  -0.45273785
  -0.70573684  1.03750043]]

First five rows of normalized X_test:
[[ 1.85442582 -1.10865585 -0.90968766 -0.6847

# **Functions**

In [10]:
# FIX B1: helper was used in this notebook but never defined (NameError after the first model).
# Copied unchanged from the other notebooks of this project.
def _ensure_excel_file(path):
    import os
    import pandas as pd
    if not os.path.exists(path):
        parent = os.path.dirname(path)
        if parent:
            os.makedirs(parent, exist_ok=True)
        pd.DataFrame().to_excel(path)
    return path

import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def plot_best_scores(best_scores_ran, excel_file_path):
    # Extract the best pruner for each model based on RMSE and correlation coefficient
    best_rmse_scores = {}
    best_corr_coef_scores = {}

    for (model_name, pruner_name), scores in best_scores_ran.items():
        # Initialize if not already present
        if model_name not in best_rmse_scores:
            best_rmse_scores[model_name] = (scores['test_rmse'], pruner_name)
        if model_name not in best_corr_coef_scores:
            best_corr_coef_scores[model_name] = (scores['test_corr_coef'], pruner_name)

        # Update if better scores are found
        if scores['test_rmse'] < best_rmse_scores[model_name][0]:
            best_rmse_scores[model_name] = (scores['test_rmse'], pruner_name)
        if scores['test_corr_coef'] > best_corr_coef_scores[model_name][0]:
            best_corr_coef_scores[model_name] = (scores['test_corr_coef'], pruner_name)

    # Prepare data for plotting
    model_names_rmse = [f"{model} ({pruner})" for model, (rmse, pruner) in best_rmse_scores.items()]
    rmse_values = [rmse for rmse, _ in best_rmse_scores.values()]

    model_names_corr = [f"{model} ({pruner})" for model, (corr, pruner) in best_corr_coef_scores.items()]
    corr_values = [corr for corr, _ in best_corr_coef_scores.values()]

    # Plot RMSE
    plt.figure(figsize=(12, 6))
    bars_rmse = plt.bar(model_names_rmse, rmse_values, color='skyblue')

    # Highlight the best model
    best_rmse_index = np.argmin(rmse_values)
    bars_rmse[best_rmse_index].set_color('orange')

    # Annotate the bars with the RMSE scores
    for i, bar in enumerate(bars_rmse):
        plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() - 0.05,
                 f'{rmse_values[i]:.5f}', ha='center', va='bottom', color='black')

    # Add labels and title for the RMSE bar chart
    plt.xlabel('Model (Pruner)')
    plt.ylabel('Test RMSE')
    plt.title('Best Test RMSE for Each Model')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    rmse_image_path = 'rmse_plot.png'
    _ensure_parent_dir(rmse_image_path)
    plt.savefig(_ensure_parent_dir(rmse_image_path))
    plt.close()

    # Plot Correlation Coefficient
    plt.figure(figsize=(12, 6))
    bars_corr = plt.bar(model_names_corr, corr_values, color='lightgreen')

    # Highlight the best model
    best_corr_index = np.argmax(corr_values)
    bars_corr[best_corr_index].set_color('orange')

    # Annotate the bars with the correlation coefficient scores
    for i, bar in enumerate(bars_corr):
        plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() - 0.05,
                 f'{corr_values[i]:.5f}', ha='center', va='bottom', color='black')

    # Add labels and title for the correlation coefficient bar chart
    plt.xlabel('Model (Pruner)')
    plt.ylabel('Correlation Coefficient')
    plt.title('Best Correlation Coefficient for Each Model')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    corr_image_path = 'corr_plot.png'
    _ensure_parent_dir(corr_image_path)
    plt.savefig(_ensure_parent_dir(corr_image_path))
    plt.close()

    # Load the existing Excel file
    workbook = load_workbook(_ensure_excel_file(excel_file_path))

    # Create a new sheet for the plots
    sheet_name = 'Best Models Plots'
    if sheet_name in workbook.sheetnames:
        sheet = workbook[sheet_name]
    else:
        sheet = workbook.create_sheet(title=sheet_name)

    # Insert images into the new Excel sheet
    img_rmse = Image(rmse_image_path)
    img_corr = Image(corr_image_path)

    # Insert images
    sheet.add_image(img_rmse, 'A1')
    sheet.add_image(img_corr, 'A20')  # Adjust the position as needed

    # Save the workbook
    _ensure_parent_dir(excel_file_path)
    workbook.save(_ensure_parent_dir(excel_file_path))

    # Clean up the image files
    if os.path.exists(str(rmse_image_path)): os.remove(str(rmse_image_path))
    if os.path.exists(str(corr_image_path)): os.remove(str(corr_image_path))

# Example usage
# plot_best_scores(best_scores_ran, 'path_to_your_excel_file.xlsx')

In [11]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def generate_interpretml_explanations_summary_pruners(
    results_dict, X_train, y_train, X_test, feature_names, instance_indices=None, excel_file_path=None
):
    if instance_indices is None:
        instance_indices = range(len(X_test))
    elif isinstance(instance_indices, int):
        instance_indices = [instance_indices]

    valid_indices = [idx for idx in instance_indices if 0 <= idx < len(X_test)]
    if not valid_indices:
        print("No valid instance indices provided.")
        return

    if isinstance(X_test, pd.DataFrame):
        instances_to_explain = X_test.iloc[valid_indices]
    else:
        instances_to_explain = X_test[valid_indices]

    best_model_pruners = {}
    for model_key, model_info in results_dict.items():
        if isinstance(model_key, tuple):
            model_name, pruner_name = model_key
        else:
            model_name = model_key
            pruner_name = None

        best_score = model_info.get('best_score')
        if best_score is None:
            print(f"No 'best_score' found for {model_key}. Skipping this combination.")
            continue

        if model_name not in best_model_pruners:
            best_model_pruners[model_name] = {
                'pruner_name': pruner_name,
                'model_info': model_info,
                'best_score': best_score
            }
        else:
            current_best_score = best_model_pruners[model_name]['best_score']
            if best_score < current_best_score:
                best_model_pruners[model_name] = {
                    'pruner_name': pruner_name,
                    'model_info': model_info,
                    'best_score': best_score
                }

    for model_name, info in best_model_pruners.items():
        pruner_name = info['pruner_name']
        model_info = info['model_info']
        best_params = dict(model_info['best_params'])  # don't mutate original!
        model_class = model_classes.get(model_name)

        if model_class is None:
            print(f"Model {model_name} is not supported or not available.")
            continue

        if model_name == 'CatBoost':
            best_params['verbose'] = 0

        # ------- Main model fit logic ---------
        if model_name == "TabNet":
            # TabNet: reshape y, fit, flatten pred for LIME/SHAP, etc.
            y_train_tabnet = np.array(y_train).reshape(-1, 1)
            try:
                model = model_class(**{k: v for k, v in best_params.items() if k != "verbose"})
            except TypeError:
                model = model_class()
            model.fit(np.array(X_train), y_train_tabnet, max_epochs=100, patience=10, batch_size=1024, eval_set=[(np.array(X_train), y_train_tabnet)])
            def predict_fn(data):
                preds = model.predict(np.array(data))
                # flatten for interpreters
                return preds.flatten()
        else:
            try:
                model = model_class(**best_params)
            except TypeError:
                model = model_class()
            model.fit(X_train, y_train)
            def predict_fn(data):
                return model.predict(data)

        if isinstance(X_train, pd.DataFrame):
            data_for_explainer = X_train.values
        else:
            data_for_explainer = X_train

        if isinstance(instances_to_explain, pd.DataFrame):
            data_for_explanation = instances_to_explain.values
        else:
            data_for_explanation = instances_to_explain

        # Generate LIME explanations
        lime_explainer = LimeTabular(
            predict_fn,
            data=data_for_explainer,
            feature_names=feature_names,
            random_state=1,
            mode='regression'
        )
        lime_explanation = lime_explainer.explain_local(data_for_explanation)

        feature_importances_lime = {}
        num_instances = len(valid_indices)
        for idx in range(num_instances):
            explanation = lime_explanation.data(idx)
            for feature_name, feature_score in zip(explanation['names'], explanation['scores']):
                feature_importances_lime[feature_name] = feature_importances_lime.get(feature_name, 0) + abs(feature_score)
        feature_importances_lime = {k: v / num_instances for k, v in feature_importances_lime.items()}
        feature_importances_lime = {k: round(v, 3) for k, v in feature_importances_lime.items()}

        # Generate SHAP explanations using ShapKernel
        try:
            shap_explainer = ShapKernel(predict_fn, data_for_explainer, feature_names=feature_names)
            shap_explanation = shap_explainer.explain_local(data_for_explanation)

            feature_importances_shap = {}
            for idx in range(num_instances):
                explanation = shap_explanation.data(idx)
                for feature_name, feature_score in zip(explanation['names'], explanation['scores']):
                    feature_importances_shap[feature_name] = feature_importances_shap.get(feature_name, 0) + abs(feature_score)

            feature_importances_shap = {k: v / num_instances for k, v in feature_importances_shap.items()}
            feature_importances_shap = {k: round(v, 3) for k, v in feature_importances_shap.items()}
        except Exception as e:
            print(f"Could not compute SHAP values for model {model_name}: {e}")
            feature_importances_shap = {}

        # Plot LIME and SHAP feature importances side by side
        fig, axes = plt.subplots(1, 2, figsize=(34, 36))

        # Plot LIME feature importances
        lime_importances_df = pd.DataFrame.from_dict(
            feature_importances_lime, orient='index', columns=['importance']
        )
        lime_importances_df.sort_values(by='importance', ascending=False, inplace=True)
        lime_importances_df.plot(kind='bar', legend=False, color='skyblue', ax=axes[0])
        axes[0].set_title(f"LIME Feature Importances for {model_name}")
        axes[0].set_ylabel("Average Absolute Importance Score")
        axes[0].set_xlabel("Features")
        axes[0].tick_params(axis='x', rotation=45)

        for p in axes[0].patches:
            height = p.get_height()
            axes[0].annotate(f'{height:.3f}',
                             (p.get_x() + p.get_width() / 2., height),
                             ha='center', va='bottom', fontsize=8)

        # Plot SHAP feature importances
        shap_importances_df = pd.DataFrame.from_dict(
            feature_importances_shap, orient='index', columns=['importance']
        )
        shap_importances_df.sort_values(by='importance', ascending=False, inplace=True)
        shap_importances_df.plot(kind='bar', legend=False, color='orange', ax=axes[1])
        axes[1].set_title(f"SHAP Feature Importances for {model_name}")
        axes[1].set_ylabel("Average Absolute SHAP Value")
        axes[1].set_xlabel("Features")
        axes[1].tick_params(axis='x', rotation=45)

        for p in axes[1].patches:
            height = p.get_height()
            axes[1].annotate(f'{height:.3f}',
                             (p.get_x() + p.get_width() / 2., height),
                             ha='center', va='bottom', fontsize=8)

        plt.tight_layout()

        # Save plots as images
        image_path = f'feature_importances_{model_name}.png'
        fig.savefig(_ensure_parent_dir(image_path))
        plt.close(fig)

        # Optionally insert images and scores into an Excel file
        if excel_file_path:
            workbook = load_workbook(_ensure_excel_file(excel_file_path))
            sheet_name = f'{model_name} Explanations'
            if sheet_name in workbook.sheetnames:
                sheet = workbook[sheet_name]
            else:
                sheet = workbook.create_sheet(title=sheet_name)

            # Insert images into the new Excel sheet
            img = Image(image_path)
            sheet.add_image(img, 'A1')

            # Create a new sheet for feature importance scores
            scores_sheet_name = f'{model_name} Scores'
            if scores_sheet_name in workbook.sheetnames:
                scores_sheet = workbook[scores_sheet_name]
            else:
                scores_sheet = workbook.create_sheet(title=scores_sheet_name)

            # Write LIME scores
            scores_sheet.append(['Feature', 'LIME Importance'])
            for feature, importance in feature_importances_lime.items():
                scores_sheet.append([feature, importance])

            # Write SHAP scores if available
            if feature_importances_shap:
                scores_sheet.append(['Feature', 'SHAP Importance'])
                for feature, importance in feature_importances_shap.items():
                    scores_sheet.append([feature, importance])

            # Save the workbook
            _ensure_parent_dir(excel_file_path)
            workbook.save(_ensure_parent_dir(excel_file_path))

            # Clean up the image file
            if os.path.exists(str(image_path)): os.remove(str(image_path))

# **Hyperparameter tuning using Autosampler by Optuna**

In [12]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
import joblib

def mseloss_objective(yhat, y, sample_weight=None):
    if not torch.is_tensor(yhat):
        yhat = torch.from_numpy(np.array(yhat)).float()
    if not torch.is_tensor(y):
        y = torch.from_numpy(np.array(y)).float()
    gradient = yhat - y
    hessian = torch.ones_like(yhat)
    return gradient, hessian


def rmseloss_metric(yhat, y, sample_weight=None):
    if not torch.is_tensor(yhat):
        yhat = torch.from_numpy(np.array(yhat)).float()
    if not torch.is_tensor(y):
        y = torch.from_numpy(np.array(y)).float()
    loss = torch.sqrt(torch.mean((yhat - y) ** 2))
    return loss


def hyperparameter_tuning_all(X_train, y_train, X_test, y_test, excel_path):

    X_train = np.array(X_train)
    y_train = np.array(y_train)
    X_test = np.array(X_test)
    y_test = np.array(y_test)

    # ================= NEW =================
    model_save_dir = "./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/hyperparameter_tuning/models"
    os.makedirs(model_save_dir, exist_ok=True)
    best_rmse_tracker = {}
    # =======================================

    models = {
        'Random Forest': (RandomForestRegressor, {
            'n_estimators': [100, 200, 300, 500, 700],
            'criterion': ['squared_error', 'absolute_error', 'friedman_mse', 'poisson'],
            'max_depth': [None, 10, 20, 30, 40],
            'min_samples_split': [2, 5, 10, 0.01],
            'min_samples_leaf': [1, 3, 5, 0.01],
            'min_weight_fraction_leaf': [0.0, 0.01, 0.1, 0.2],
            'max_features': [1.0, 'sqrt', 'log2', 0.3, 0.5],
            'max_leaf_nodes': [None, 50, 100, 200],
            'min_impurity_decrease': [0.0, 0.01, 0.1, 0.2],
            'n_jobs': [-1],
            'random_state': [42],
            'verbose': [0],
            'warm_start': [False],
            'ccp_alpha': [0.0, 0.001, 0.01, 0.05, 0.1]
        }),
        'Gradient Boosting': (GradientBoostingRegressor, {
            'loss': ['squared_error', 'absolute_error', 'huber'],  # FIX 3: 'quantile' removed (point model must target the mean)
            'learning_rate': [0.01, 0.05, 0.1, 0.2],
            'n_estimators': [100, 200, 300, 500, 700],
            'subsample': [1.0, 0.9, 0.7, 0.5],
            'criterion': ['friedman_mse', 'squared_error'],
            'min_samples_split': [2, 5, 10, 0.01],
            'min_samples_leaf': [1, 3, 5, 0.01],
            'min_weight_fraction_leaf': [0.0, 0.01, 0.05, 0.1],
            'max_depth': [3, 5, 7, 10],
            'min_impurity_decrease': [0.0, 0.01, 0.1],
            'init': [None],
            'random_state': [42],
            'max_features': [None, 'sqrt', 'log2', 0.5],
            'alpha': [0.9, 0.5, 0.1],
            'verbose': [0],
            'max_leaf_nodes': [None, 10, 30, 50],
            'warm_start': [False],
            'validation_fraction': [0.1],
            'n_iter_no_change': [None, 10, 20],
            'tol': [1e-4, 1e-3],
            'ccp_alpha': [0.0, 0.001, 0.01]
        }),
        'XGBoost': (XGBRegressor, {
            'n_estimators': [100, 200, 300, 400, 500],
            'learning_rate': [0.01, 0.05, 0.1, 0.15],
            'max_depth': [3, 5, 7],
            'min_child_weight': [1, 3, 5],
            'gamma': [0, 0.1, 0.5, 1],
            'subsample': [0.5, 0.6, 0.7, 0.8, 0.9],
            'colsample_bytree': [0.5, 0.7, 0.9],
            'colsample_bylevel': [0.5, 0.7, 0.9],
            'reg_alpha': [0, 0.01, 0.1, 1],
            'reg_lambda': [0.1, 1, 5, 10],
            'objective': ['reg:squarederror'],
            'random_state': [42],
            'n_jobs': [-1]
        }),
        'LightGBM': (LGBMRegressor, {
            'n_estimators': [100, 200, 300, 400, 500],
            'learning_rate': [0.01, 0.05, 0.1, 0.15],
            'num_leaves': [15, 31, 63],
            'max_depth': [3, 5, 7, -1],
            'min_child_samples': [1, 5, 10, 20],
            'subsample': [0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
            'colsample_bytree': [0.5, 0.7, 0.9, 1.0],
            'reg_alpha': [0, 0.01, 0.1, 1],
            'reg_lambda': [0, 0.1, 1, 10],
            'min_child_weight': [1e-5, 1e-3, 1e-2, 1e-1],
            'bagging_freq': [0, 1, 5],
            'objective': ['regression'],
            'random_state': [42],
            'n_jobs': [-1],
            'verbose': [-1]
        }),
        'GPBoost': (GPBoostRegressor, {
            'n_estimators': [100, 200, 300, 400, 500],
            'learning_rate': [0.01, 0.05, 0.1, 0.15],
            'max_depth': [3, 5, 7, -1],
            'num_leaves': [15, 31, 63],
            'min_child_samples': [1, 5, 10, 20],
            'subsample': [0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
            'colsample_bytree': [0.5, 0.7, 0.9, 1.0],
            'reg_alpha': [0, 0.1, 0.5, 1.0],
            'reg_lambda': [0, 0.1, 0.5, 1.0],
            'min_child_weight': [1e-5, 1e-3, 1e-2, 1e-1],
            'random_state': [42],
            'n_jobs': [-1],
            'verbose': [-1]
        }),
        'CatBoost': (CatBoostRegressor, {
            'iterations': [200, 500, 1000],
            'learning_rate': [0.01, 0.03, 0.05, 0.1],
            'depth': [4, 6, 8, 10],
            'l2_leaf_reg': [1, 3, 5, 7, 9],
            'border_count': [32, 64, 128],
            'min_data_in_leaf': [1, 5, 10, 20],
            'rsm': [0.6, 0.8, 1.0],
            'bagging_temperature': [0, 1, 10],
            'random_seed': [42],
            'verbose': [0]
        }),
        'NGBoost': (NGBRegressor, {
            'n_estimators': [200, 500, 1000],
            'learning_rate': [0.01, 0.03, 0.05, 0.1],
            'natural_gradient': [True, False],
            'minibatch_frac': [0.5, 0.7, 0.9, 1.0],
            'col_sample': [0.5, 0.7, 0.9, 1.0],
            'Dist': [Normal],
            'Score': [LogScore],
            'random_state': [42],
            'verbose': [0]
        }),
        'TabNet': (TabNetRegressor, {
            'n_d': [8, 16, 32, 64],
            'n_a': [8, 16, 32, 64],
            'n_steps': [3, 5, 7, 10],
            'gamma': [1.0, 1.3, 1.5, 2.0],
            'lambda_sparse': [1e-4, 1e-3, 1e-2],
            'optimizer_params': [{'lr': 2e-2}], # Fixed learning rate as recommended
            'mask_type': ['sparsemax', 'entmax'],
            'n_shared': [1, 2, 3],
            'n_independent': [1, 2, 3],
            'scheduler_params': [{"step_size": 10, "gamma": 0.9}],
            'scheduler_fn': [torch.optim.lr_scheduler.StepLR],
            'seed': [42],
            'verbose': [0]
        }),
        'HistGradientBoosting': (HistGradientBoostingRegressor, {
            'learning_rate': [0.01, 0.05, 0.1, 0.15],
            'max_iter': [100, 200, 300, 400, 500],
            'max_depth': [3, 5, 7, None],
            'min_samples_leaf': [5, 10, 20],
            'max_leaf_nodes': [15, 31, 63, None],
            'l2_regularization': [0.0, 0.1, 0.5, 1.0],
            'max_bins': [64, 128, 255],
            'early_stopping': [True, False],
            'validation_fraction': [0.1, 0.2],
            'n_iter_no_change': [5, 10, 15],
            'loss': ['squared_error'],
            'random_state': [42],
            'verbose': [0]
        }),
        'PGBM': (PGBM, {})
    }

    pruners = [
        optuna.pruners.MedianPruner(),
        optuna.pruners.NopPruner(),
        optuna.pruners.PatientPruner(optuna.pruners.MedianPruner(), patience=3),
        optuna.pruners.PercentilePruner(25.0),
        optuna.pruners.SuccessiveHalvingPruner(),
        optuna.pruners.HyperbandPruner(),
        optuna.pruners.ThresholdPruner(upper=float(np.var(y_train))),  # FIX 2: was lower=0.1 (pruned the BEST trials); now prunes trials worse than predicting the mean
        optuna.pruners.WilcoxonPruner()
    ]

    best_scores = {}
    predictions_df = pd.DataFrame({'Actual': y_test})
    timing_records = []
    pruning_records = []  # FIX 2: completed / pruned trial counts per model x pruner (new Excel sheet)

    # FIX 1: hyper-parameters are scored by 5-fold CV on the TRAINING data only.
    # The test set is used once, after tuning, to evaluate the chosen model.
    cv_splits = list(KFold(n_splits=5, shuffle=True, random_state=42).split(X_train))

    def _fit_model(model_name, model_class, params, X_fit, y_fit):
        if model_name == 'PGBM':
            m = PGBM()
            m.train((X_fit, y_fit), objective=mseloss_objective, metric=rmseloss_metric, params=params)
        else:
            m = model_class(**params)
            if model_name == 'TabNet':
                m.fit(X_fit, y_fit.reshape(-1, 1))
            else:
                m.fit(X_fit, y_fit)
        return m

    def _predict(model_name, m, X):
        p = m.predict(X)
        if torch.is_tensor(p):
            p = p.detach().cpu().numpy()
        return np.asarray(p).ravel()

    def _cv_mse(trial, model_name, model_class, params, pruner_name):
        fold_mse = []
        for k, (tr_idx, va_idx) in enumerate(cv_splits):
            m = _fit_model(model_name, model_class, params, X_train[tr_idx], y_train[tr_idx])
            fold_mse.append(mean_squared_error(y_train[va_idx], _predict(model_name, m, X_train[va_idx])))
            # FIX 2: report progress after every fold so the pruner can stop bad trials early.
            # WilcoxonPruner compares per-fold values; the others use the running mean.
            trial.report(fold_mse[-1] if pruner_name == 'WilcoxonPruner' else float(np.mean(fold_mse)), k)
            if trial.should_prune():
                rmse_trial_history[pruner_name].append(np.nan)
                raise optuna.TrialPruned()
        return float(np.mean(fold_mse))

    def _keep_if_best(model_name, model_class, params, pruner_name, rmse):
        # ===== SAVE BEST MODEL (best CV score), refit on the FULL training set =====
        if rmse < best_rmse_tracker[(model_name, pruner_name)]:
            best_rmse_tracker[(model_name, pruner_name)] = rmse
            save_path = os.path.join(model_save_dir, f"{model_name}_{pruner_name}_BEST.pkl")
            _ensure_parent_dir(save_path)
            joblib.dump(_fit_model(model_name, model_class, params, X_train, y_train), _ensure_parent_dir(save_path))

    for model_name, (model_class, param_space) in models.items():

        rmse_trial_history = {p.__class__.__name__: [] for p in pruners}

        for pruner in pruners:
            pruner_name = pruner.__class__.__name__
            print(f"Running Optuna for {model_name} with {pruner_name}...")
            start_time = time.time()

            best_rmse_tracker[(model_name, pruner_name)] = np.inf

            sampler = optunahub.load_module("samplers/auto_sampler").AutoSampler(seed=42)  # FIX 2: seeded -> reproducible
            study = optuna.create_study(direction='minimize', sampler=sampler, pruner=pruner)

            if model_name == 'PGBM':

                def pgbm_objective(trial):
                    params = {
                            'n_estimators': trial.suggest_categorical('n_estimators', [100, 200, 300, 500]),
                            'learning_rate': trial.suggest_categorical('learning_rate', [0.01, 0.05, 0.1, 0.15]),
                            'max_leaves': trial.suggest_int('max_leaves', 15, 63),
                            'min_split_gain': trial.suggest_categorical('min_split_gain', [0.0, 0.1, 0.5, 1.0]),
                            'reg_lambda': trial.suggest_categorical('reg_lambda', [0.1, 1.0, 5.0, 10.0]),
                            'feature_fraction': trial.suggest_categorical('feature_fraction', [0.5, 0.7, 0.9, 1.0]),
                            'bagging_fraction': trial.suggest_categorical('bagging_fraction', [0.5, 0.7, 0.9, 1.0]),
                            'tree_correlation': trial.suggest_categorical('tree_correlation', [0.0, 0.1, 0.2, 0.3]),
                            'min_data_in_leaf': trial.suggest_categorical('min_data_in_leaf', [3, 5, 10, 20]),
                            'max_bin': trial.suggest_categorical('max_bin', [64, 128, 256]),
                            'distribution': trial.suggest_categorical('distribution', ['normal', 'studentt', 'laplace']),
                            'objective': 'mse',
                            'metric': 'rmse',
                            'random_state': 42,
                            'verbose': 0
                        }

                    # FIX 1: CV on training data (was: fit on train, score on TEST)
                    mse = _cv_mse(trial, model_name, None, params, pruner_name)
                    rmse = np.sqrt(mse)

                    rmse_trial_history[pruner_name].append(rmse)
                    _keep_if_best(model_name, None, params, pruner_name, rmse)

                    return mse

                study.optimize(pgbm_objective, n_trials=50)

            else:

                def objective(trial):
                    params = {}
                    for key, values in param_space.items():
                        params[key] = trial.suggest_categorical(key, values)

                    # FIX 1: CV on training data (was: fit on train, score on TEST)
                    mse = _cv_mse(trial, model_name, model_class, params, pruner_name)
                    rmse = np.sqrt(mse)

                    rmse_trial_history[pruner_name].append(rmse)
                    _keep_if_best(model_name, model_class, params, pruner_name, rmse)

                    return mse

                study.optimize(objective, n_trials=50)

            elapsed_time = time.time() - start_time

            # FIX B13 (non-math): if the pruner stopped EVERY trial, no model was saved and the load below
            # crashed (FileNotFoundError). Record the pair as "all trials pruned" and skip it; each model
            # still has NopPruner (never prunes), so the other notebooks always find params for it.
            if not any(t.state == optuna.trial.TrialState.COMPLETE for t in study.trials):
                print(f"  {model_name} / {pruner_name}: all {len(study.trials)} trials pruned - pair skipped")
                pruning_records.append({
                    'Model': model_name, 'Pruner': pruner_name,
                    'Trials': len(study.trials), 'Completed': 0, 'Pruned': len(study.trials),
                    'Pruned_%': 100.0, 'Best_CV_RMSE': np.nan, 'Test_RMSE': np.nan,
                    'Tuning_Time_Seconds': round(elapsed_time, 1),
                })
                timing_records.append({'Model': model_name, 'Pruner': pruner_name, 'Tuning_Time_Seconds': elapsed_time})
                continue

            # Load frozen model (NO RETRAIN)
            best_model = joblib.load(
                os.path.join(model_save_dir, f"{model_name}_{pruner_name}_BEST.pkl")
            )

            y_pred = _predict(model_name, best_model, X_test)  # test set used ONCE, here

            mse = mean_squared_error(y_test, y_pred)
            rmse = np.sqrt(mse)
            corr_coef = np.corrcoef(y_test, y_pred)[0, 1]


            predictions_df[f'{model_name}_{pruner_name}_Predicted'] = y_pred

            # FIX 2: pruning statistics for the new Excel sheet
            n_complete = sum(t.state == optuna.trial.TrialState.COMPLETE for t in study.trials)
            n_pruned = sum(t.state == optuna.trial.TrialState.PRUNED for t in study.trials)
            pruning_records.append({
                'Model': model_name, 'Pruner': pruner_name,
                'Trials': len(study.trials), 'Completed': n_complete, 'Pruned': n_pruned,
                'Pruned_%': round(100.0 * n_pruned / max(len(study.trials), 1), 1),
                'Best_CV_RMSE': float(np.sqrt(study.best_value)),
                'Test_RMSE': float(rmse),
                'Tuning_Time_Seconds': round(elapsed_time, 1),
            })

            best_scores[(model_name, pruner_name)] = {
                'best_score': study.best_value,  # FIX 1: CV MSE on train (was TEST MSE)
                'cv_rmse': float(np.sqrt(study.best_value)),
                'best_params': study.best_params,
                'test_mse': mse,
                'test_rmse': rmse,
                'test_corr_coef': corr_coef,
                'pruner': pruner_name
            }

            timing_records.append({
                'Model': model_name,
                'Pruner': pruner_name,
                'Tuning_Time_Seconds': elapsed_time
            })

        # RMSE plots & Excel writing (UNCHANGED)
        rmse_df = pd.DataFrame(rmse_trial_history)
        rmse_df.insert(0, "Trial", np.arange(1, len(rmse_df) + 1))

        if not os.path.exists(excel_path):
            pd.DataFrame().to_excel(excel_path)
        with pd.ExcelWriter(_ensure_parent_dir(_ensure_excel_file(excel_path)), engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
            writer
            rmse_df.to_excel(writer, sheet_name=f"RMSE_Trials_{model_name}", index=False)
        # ================= SAVE RMSE PLOT =================
        plot_dir = os.path.dirname(excel_path)
        plot_path = os.path.join(plot_dir, f"RMSE_Trials_{model_name}.png")

        plt.figure(figsize=(10, 6))
        for pruner_name, values in rmse_trial_history.items():
            if len(values) > 0:   # <-- important safety check
                plt.plot(values, label=pruner_name, linewidth=2)

        plt.title(f"RMSE Variation Over Trials\n{model_name}")
        plt.xlabel("Trial Number")
        plt.ylabel("RMSE")
        plt.legend(loc="center left", bbox_to_anchor=(1.02, 0.5))
        plt.tight_layout()

        _ensure_parent_dir(plot_path)
        plt.savefig(_ensure_parent_dir(plot_path), dpi=100, bbox_inches="tight")
        plt.close()

        # ================= INSERT PLOT INTO EXCEL =================
        wb = load_workbook(_ensure_excel_file(excel_path))
        ws = wb[f"RMSE_Trials_{model_name}"]

        img = Image(plot_path)
        img.anchor = "J2"
        ws.add_image(img)

        _ensure_parent_dir(excel_path)
        wb.save(_ensure_parent_dir(excel_path))

    timing_df = pd.DataFrame(timing_records)

    if not os.path.exists(excel_path):
        pd.DataFrame().to_excel(excel_path)
    with pd.ExcelWriter(_ensure_parent_dir(_ensure_excel_file(excel_path)), engine='openpyxl', mode='a') as writer:
        writer
        predictions_df.to_excel(writer, sheet_name='Predictions', index=False)
        writer
        timing_df.to_excel(writer, sheet_name='Tuning_Time', index=False)
        pd.DataFrame(pruning_records).to_excel(writer, sheet_name='Pruning_Summary', index=False)  # FIX 2

    return best_scores


best_scores_autosampler = hyperparameter_tuning_all(X_train, y_train, X_test, y_test, "./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/hyperparameter_tuning/test.xlsx")


Running Optuna for Random Forest with MedianPruner...


[I 2026-11-22 10:01:35,342] A new study created in memory with name: no-name-48235a98-4d86-4de1-addc-2f65caabb338


[I 2026-11-22 10:01:37,949] Trial 0 finished with value: 911.8984467314561 and parameters: {'n_estimators': 200, 'criterion': 'squared_error', 'max_depth': 20, 'min_samples_split': 0.01, 'min_samples_leaf': 3, 'min_weight_fraction_leaf': 0.0, 'max_features': 1.0, 'max_leaf_nodes': 200, 'min_impurity_decrease': 0.1, 'n_jobs': -1, 'random_state': 42, 'verbose': 0, 'warm_start': False, 'ccp_alpha': 0.1}. Best is trial 0 with value: 911.8984467314561.


[I 2026-11-22 10:01:41,159] Trial 1 finished with value: 1022.680817045802 and parameters: {'n_estimators': 300, 'criterion': 'friedman_mse', 'max_depth': None, 'min_samples_split': 0.01, 'min_samples_leaf': 3, 'min_weight_fraction_leaf': 0.0, 'max_features': 'sqrt', 'max_leaf_nodes': 200, 'min_impurity_decrease': 0.0, 'n_jobs': -1, 'random_state': 42, 'verbose': 0, 'warm_start': False, 'ccp_alpha': 0.05}. Best is trial 0 with value: 911.8984467314561.


[I 2026-11-22 10:01:50,356] Trial 2 finished with value: 890.1411130344992 and parameters: {'n_estimators': 700, 'criterion': 'squared_error', 'max_depth': 20, 'min_samples_split': 2, 'min_samples_leaf': 0.01, 'min_weight_fraction_leaf': 0.1, 'max_features': 0.5, 'max_leaf_nodes': 200, 'min_impurity_decrease': 0.1, 'n_jobs': -1, 'random_state': 42, 'verbose': 0, 'warm_start': False, 'ccp_alpha': 0.01}. Best is trial 2 with value: 890.1411130344992.


[I 2026-11-22 10:01:57,769] Trial 3 finished with value: 1118.8462501880756 and parameters: {'n_estimators': 700, 'criterion': 'friedman_mse', 'max_depth': None, 'min_samples_split': 10, 'min_samples_leaf': 1, 'min_weight_fraction_leaf': 0.1, 'max_features': 'log2', 'max_leaf_nodes': 200, 'min_impurity_decrease': 0.1, 'n_jobs': -1, 'random_state': 42, 'verbose': 0, 'warm_start': False, 'ccp_alpha': 0.01}. Best is trial 2 with value: 890.1411130344992.


... (5,782 outputs left out)

Training on CPU


Training on CPU


Training on CPU


[I 2026-11-22 15:35:38,147] Trial 49 finished with value: 742.7649233473487 and parameters: {'n_estimators': 500, 'learning_rate': 0.05, 'max_leaves': 44, 'min_split_gain': 0.0, 'reg_lambda': 0.1, 'feature_fraction': 1.0, 'bagging_fraction': 0.5, 'tree_correlation': 0.2, 'min_data_in_leaf': 5, 'max_bin': 64, 'distribution': 'laplace'}. Best is trial 36 with value: 605.6693689759791.


In [13]:
best_scores_autosampler

{('Random Forest', 'MedianPruner'): {'best_score': 815.3942506476026,
  'cv_rmse': 28.555109011306584,
  'best_params': {'n_estimators': 200,
   'criterion': 'squared_error',
   'max_depth': 20,
   'min_samples_split': 0.01,
   'min_samples_leaf': 1,
   'min_weight_fraction_leaf': 0.0,
   'max_features': 0.5,
   'max_leaf_nodes': 200,
   'min_impurity_decrease': 0.1,
   'n_jobs': -1,
   'random_state': 42,
   'verbose': 0,
   'warm_start': False,
   'ccp_alpha': 0.1},
  'test_mse': 517.6088147057782,
  'test_rmse': np.float64(22.751017882850388),
  'test_corr_coef': np.float64(0.9547680128224817),
  'pruner': 'MedianPruner'},
 ('Random Forest', 'NopPruner'): {'best_score': 800.6499765894229,
  'cv_rmse': 28.295758985922657,
  'best_params': {'n_estimators': 300,
   'criterion': 'squared_error',
   'max_depth': 20,
   'min_samples_split': 0.01,
   'min_samples_leaf': 0.01,
   'min_weight_fraction_leaf': 0.0,
   'max_features': 0.5,
   'max_leaf_nodes': None,
   'min_impurity_decrease': 

In [14]:
# FIX K2: save the tuning results so the other notebooks LOAD them instead of copy-pasting.
# best_params.pkl  -> loaded by the other notebooks (keeps Python objects such as NGBoost's Normal)
# best_params.json -> the same content in readable form, for checking by eye
import pickle, json
_out_dir = "./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/hyperparameter_tuning"
os.makedirs(_out_dir, exist_ok=True)
with open(os.path.join(_out_dir, "best_params.pkl"), "wb") as f:
    pickle.dump(best_scores_autosampler, f)
with open(os.path.join(_out_dir, "best_params.json"), "w") as f:
    json.dump({f"{m} | {p}": v for (m, p), v in best_scores_autosampler.items()}, f, indent=1, default=str)
print("Saved tuning results to", _out_dir)


Saved tuning results to ./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/hyperparameter_tuning


# **Best Model Analysis**

In [15]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def get_best_models_and_predict(best_scores_autosampler, X_train, y_train, X_test, y_test, file_path):
    # Convert input data to NumPy arrays
    X_train = np.array(X_train)
    y_train = np.array(y_train)
    X_test = np.array(X_test)
    y_test = np.array(y_test)

    # Mapping for model creation based on dictionary keys
    model_mapping = {
        'Random Forest': RandomForestRegressor,
        'Gradient Boosting': GradientBoostingRegressor,
        'XGBoost': XGBRegressor,
        'LightGBM': LGBMRegressor,
        'CatBoost': CatBoostRegressor,
        'GPBoost': GPBoostRegressor,
        'NGBoost': NGBRegressor,
        'TabNet': TabNetRegressor,
        'HistGradientBoosting': HistGradientBoostingRegressor,
        'PGBM': PGBM  # PGBM is handled separately
    }

    # Dictionary to store the best model for each type
    best_models = {}

    # Iterate over the dictionary to find the best pruner for each model type
    for (model_name, pruner), params in best_scores_autosampler.items():
        current_score = params.get('best_score', np.inf)  # FIX 1: pick by CV score (was test_mse)
        if model_name not in best_models or current_score < best_models[model_name]['score']:
            best_models[model_name] = {
                'score': current_score,
                'params': params['best_params'],
                'pruner': pruner
            }

    # Prepare a DataFrame to store predictions
    df = pd.read_csv(file_path)

    # Iterate over the best models to train and predict
    for model_name, model_info in best_models.items():
        best_params = model_info['params']
        model_class = model_mapping.get(model_name)

        if model_class is None:
            print(f"Model {model_name} is not supported or not available.")
            continue

        # Handle specific parameters or settings for model if needed
        if model_name == 'CatBoost':
            best_params.pop('verbose', None)  # Remove 'verbose' for CatBoost

        # Create an instance of the best model with the best parameters
        if model_name == 'PGBM':
            model = model_class()
            model.train((X_train, y_train), objective=mseloss_objective, metric=rmseloss_metric, params=best_params)
            predictions = model.predict(X_test)
        elif model_name == 'TabNet':
            model = model_class(**best_params)
            model.fit(X_train, y_train.reshape(-1, 1))
            predictions = model.predict(X_test)
            predictions = predictions.ravel()
        else:
            model = model_class(**best_params)
            model.fit(X_train, y_train)
            predictions = model.predict(X_test)

        # Add predictions to the DataFrame
        df[f'{model_name} Predictions'] = predictions

        # Plot actual vs. predicted
        plt.figure(figsize=(10, 6))
        plt.scatter(y_test, predictions, alpha=0.6)
        plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'k--', color='red', lw=2)
        plt.xlabel(f"Actual {TARGET_LABEL}")
        plt.ylabel(f"Predicted {TARGET_LABEL}")
        plt.title(f"Actual vs. Predicted Values ({model_name})")
        plt.grid(True)
        plt.tight_layout()

        # Save the plot temporarily
        plot_path = f'temp_plot_{model_name}.png'
        _ensure_parent_dir(plot_path)
        plt.savefig(_ensure_parent_dir(plot_path))
        plt.close()

    # ✅ NEW OUTPUT DIRECTORY
    output_dir = "./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/hyperparameter_tuning/"
    os.makedirs(output_dir, exist_ok=True)

    output_excel_path = os.path.join(
        output_dir,
        os.path.basename(file_path).replace('.csv', '_results.xlsx')
    )

    # Save predictions and plots to Excel
    with pd.ExcelWriter(_ensure_parent_dir(output_excel_path), engine='xlsxwriter') as writer:
        # Write data to Excel
        writer
        df.to_excel(writer, sheet_name='data', index=False)

        # Get the xlsxwriter objects
        workbook = writer.book

        # Insert each plot into a separate worksheet
        for model_name in best_models.keys():
            short_model_name = ''.join([word[0] for word in model_name.split()])
            sheet_name = f'{short_model_name}_Plot'

            worksheet = workbook.add_worksheet(sheet_name)
            writer.sheets[sheet_name] = worksheet
            plot_path = f'temp_plot_{model_name}.png'
            worksheet.insert_image('A1', plot_path)

    # Clean up temporary plot files
    for model_name in best_models.keys():
        if os.path.exists(str(f'temp_plot_{model_name}.png')): os.remove(str(f'temp_plot_{model_name}.png'))

    return df, best_models

# Call the function
df, best_models = get_best_models_and_predict(best_scores_autosampler, X_train, y_train, X_test, y_test, "./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/data/test.csv")

0:	learn: 53.8742548	total: 4.77ms	remaining: 2.38s
1:	learn: 51.5332515	total: 5.47ms	remaining: 1.36s
2:	learn: 49.9627948	total: 6.04ms	remaining: 1s
3:	learn: 48.1703483	total: 6.46ms	remaining: 801ms
4:	learn: 46.3102437	total: 6.8ms	remaining: 673ms
5:	learn: 44.1693203	total: 7.15ms	remaining: 589ms
6:	learn: 42.1570137	total: 7.47ms	remaining: 526ms
7:	learn: 40.0829681	total: 7.81ms	remaining: 480ms
8:	learn: 38.2812391	total: 8.1ms	remaining: 442ms
9:	learn: 36.8455304	total: 8.51ms	remaining: 417ms
10:	learn: 35.8104374	total: 8.76ms	remaining: 389ms
11:	learn: 34.4316251	total: 9.08ms	remaining: 369ms
12:	learn: 33.1177524	total: 9.37ms	remaining: 351ms
13:	learn: 32.0308629	total: 9.63ms	remaining: 334ms
14:	learn: 30.9635085	total: 9.88ms	remaining: 319ms
15:	learn: 29.9788771	total: 10.1ms	remaining: 306ms
16:	learn: 29.3480623	total: 10.4ms	remaining: 296ms
17:	learn: 28.4194480	total: 10.6ms	remaining: 285ms
18:	learn: 27.2241139	total: 10.9ms	remaining: 276ms
19:	lear

Training on CPU
Estimator 0/300, Train metric: 55.0218
Estimator 1/300, Train metric: 53.3739
Estimator 2/300, Train metric: 52.0749
Estimator 3/300, Train metric: 50.5340
Estimator 4/300, Train metric: 48.7228
Estimator 5/300, Train metric: 47.0657
Estimator 6/300, Train metric: 46.0698
Estimator 7/300, Train metric: 44.7027
Estimator 8/300, Train metric: 43.7969
Estimator 9/300, Train metric: 42.3529
Estimator 10/300, Train metric: 41.1388
Estimator 11/300, Train metric: 39.9451
Estimator 12/300, Train metric: 39.3936
Estimator 13/300, Train metric: 38.5708
Estimator 14/300, Train metric: 37.4268
Estimator 15/300, Train metric: 36.2130
Estimator 16/300, Train metric: 35.8212
Estimator 17/300, Train metric: 35.3588
Estimator 18/300, Train metric: 34.8089
Estimator 19/300, Train metric: 34.4374
Estimator 20/300, Train metric: 34.0438
Estimator 21/300, Train metric: 33.6476
Estimator 22/300, Train metric: 32.8975
Estimator 23/300, Train metric: 32.2363
Estimator 24/300, Train metric: 31

Estimator 39/300, Train metric: 22.8892
Estimator 40/300, Train metric: 22.4073
Estimator 41/300, Train metric: 21.8657
Estimator 42/300, Train metric: 21.3036
Estimator 43/300, Train metric: 21.0520
Estimator 44/300, Train metric: 20.6351
Estimator 45/300, Train metric: 20.4684
Estimator 46/300, Train metric: 20.1000
Estimator 47/300, Train metric: 19.8026
Estimator 48/300, Train metric: 19.5589
Estimator 49/300, Train metric: 19.4191
Estimator 50/300, Train metric: 19.0989
Estimator 51/300, Train metric: 18.7039
Estimator 52/300, Train metric: 18.7178
Estimator 53/300, Train metric: 18.5688
Estimator 54/300, Train metric: 18.3391
Estimator 55/300, Train metric: 17.9783
Estimator 56/300, Train metric: 17.8861
Estimator 57/300, Train metric: 17.7476
Estimator 58/300, Train metric: 17.4496
Estimator 59/300, Train metric: 17.1360
Estimator 60/300, Train metric: 16.9987
Estimator 61/300, Train metric: 16.7382
Estimator 62/300, Train metric: 16.5740
Estimator 63/300, Train metric: 16.3673


Estimator 74/300, Train metric: 14.4790
Estimator 75/300, Train metric: 14.3939
Estimator 76/300, Train metric: 14.2124
Estimator 77/300, Train metric: 14.1670
Estimator 78/300, Train metric: 14.0933
Estimator 79/300, Train metric: 13.9494
Estimator 80/300, Train metric: 13.9759
Estimator 81/300, Train metric: 13.8087
Estimator 82/300, Train metric: 13.6064
Estimator 83/300, Train metric: 13.4929
Estimator 84/300, Train metric: 13.3332
Estimator 85/300, Train metric: 13.2440
Estimator 86/300, Train metric: 13.0907
Estimator 87/300, Train metric: 13.0373
Estimator 88/300, Train metric: 13.0103
Estimator 89/300, Train metric: 12.8905
Estimator 90/300, Train metric: 12.8355
Estimator 91/300, Train metric: 12.7364
Estimator 92/300, Train metric: 12.5994
Estimator 93/300, Train metric: 12.4648
Estimator 94/300, Train metric: 12.2250
Estimator 95/300, Train metric: 12.0125
Estimator 96/300, Train metric: 11.9530
Estimator 97/300, Train metric: 11.9207
Estimator 98/300, Train metric: 11.8422


Estimator 111/300, Train metric: 10.6698
Estimator 112/300, Train metric: 10.6707
Estimator 113/300, Train metric: 10.5762
Estimator 114/300, Train metric: 10.5468
Estimator 115/300, Train metric: 10.4995
Estimator 116/300, Train metric: 10.4526
Estimator 117/300, Train metric: 10.4320
Estimator 118/300, Train metric: 10.4033
Estimator 119/300, Train metric: 10.2728
Estimator 120/300, Train metric: 10.2700
Estimator 121/300, Train metric: 10.1300
Estimator 122/300, Train metric: 10.0422
Estimator 123/300, Train metric: 9.9921
Estimator 124/300, Train metric: 9.9513
Estimator 125/300, Train metric: 9.9225
Estimator 126/300, Train metric: 9.8781
Estimator 127/300, Train metric: 9.8022
Estimator 128/300, Train metric: 9.8036
Estimator 129/300, Train metric: 9.6728
Estimator 130/300, Train metric: 9.6110
Estimator 131/300, Train metric: 9.5791
Estimator 132/300, Train metric: 9.5569
Estimator 133/300, Train metric: 9.5385
Estimator 134/300, Train metric: 9.4700
Estimator 135/300, Train met

Estimator 143/300, Train metric: 9.0502
Estimator 144/300, Train metric: 8.9818
Estimator 145/300, Train metric: 8.9173
Estimator 146/300, Train metric: 8.8772
Estimator 147/300, Train metric: 8.7642
Estimator 148/300, Train metric: 8.7541
Estimator 149/300, Train metric: 8.7641
Estimator 150/300, Train metric: 8.7435
Estimator 151/300, Train metric: 8.6394
Estimator 152/300, Train metric: 8.6179
Estimator 153/300, Train metric: 8.5644
Estimator 154/300, Train metric: 8.5225
Estimator 155/300, Train metric: 8.3490
Estimator 156/300, Train metric: 8.3194
Estimator 157/300, Train metric: 8.2799
Estimator 158/300, Train metric: 8.2496
Estimator 159/300, Train metric: 8.2563
Estimator 160/300, Train metric: 8.2039
Estimator 161/300, Train metric: 8.1449
Estimator 162/300, Train metric: 8.1309
Estimator 163/300, Train metric: 8.1263
Estimator 164/300, Train metric: 8.1300
Estimator 165/300, Train metric: 8.1188
Estimator 166/300, Train metric: 8.1135
Estimator 167/300, Train metric: 8.0945


Estimator 172/300, Train metric: 7.9138
Estimator 173/300, Train metric: 7.8723
Estimator 174/300, Train metric: 7.8243
Estimator 175/300, Train metric: 7.8278
Estimator 176/300, Train metric: 7.8433
Estimator 177/300, Train metric: 7.8214
Estimator 178/300, Train metric: 7.8264
Estimator 179/300, Train metric: 7.7773
Estimator 180/300, Train metric: 7.6972
Estimator 181/300, Train metric: 7.6182
Estimator 182/300, Train metric: 7.5688
Estimator 183/300, Train metric: 7.5420
Estimator 184/300, Train metric: 7.4684
Estimator 185/300, Train metric: 7.4722
Estimator 186/300, Train metric: 7.4772
Estimator 187/300, Train metric: 7.3982
Estimator 188/300, Train metric: 7.3940
Estimator 189/300, Train metric: 7.3914
Estimator 190/300, Train metric: 7.3487
Estimator 191/300, Train metric: 7.2955
Estimator 192/300, Train metric: 7.2783
Estimator 193/300, Train metric: 7.2857
Estimator 194/300, Train metric: 7.2736
Estimator 195/300, Train metric: 7.2801
Estimator 196/300, Train metric: 7.2204


Estimator 204/300, Train metric: 7.0687
Estimator 205/300, Train metric: 7.0643
Estimator 206/300, Train metric: 7.0131
Estimator 207/300, Train metric: 7.0047
Estimator 208/300, Train metric: 6.9527
Estimator 209/300, Train metric: 6.9280
Estimator 210/300, Train metric: 6.9172
Estimator 211/300, Train metric: 6.9223
Estimator 212/300, Train metric: 6.8699
Estimator 213/300, Train metric: 6.8710
Estimator 214/300, Train metric: 6.8053
Estimator 215/300, Train metric: 6.7838
Estimator 216/300, Train metric: 6.7737
Estimator 217/300, Train metric: 6.7405
Estimator 218/300, Train metric: 6.6536
Estimator 219/300, Train metric: 6.6429
Estimator 220/300, Train metric: 6.6378
Estimator 221/300, Train metric: 6.6014
Estimator 222/300, Train metric: 6.6011
Estimator 223/300, Train metric: 6.6056
Estimator 224/300, Train metric: 6.5337
Estimator 225/300, Train metric: 6.4972
Estimator 226/300, Train metric: 6.4960
Estimator 227/300, Train metric: 6.4901
Estimator 228/300, Train metric: 6.4128


Estimator 234/300, Train metric: 6.3246
Estimator 235/300, Train metric: 6.3318
Estimator 236/300, Train metric: 6.3256
Estimator 237/300, Train metric: 6.3282
Estimator 238/300, Train metric: 6.3271
Estimator 239/300, Train metric: 6.3365
Estimator 240/300, Train metric: 6.2886
Estimator 241/300, Train metric: 6.2900
Estimator 242/300, Train metric: 6.2581
Estimator 243/300, Train metric: 6.2621
Estimator 244/300, Train metric: 6.2624
Estimator 245/300, Train metric: 6.2135
Estimator 246/300, Train metric: 6.2100
Estimator 247/300, Train metric: 6.2147
Estimator 248/300, Train metric: 6.1478
Estimator 249/300, Train metric: 6.1009
Estimator 250/300, Train metric: 6.0913
Estimator 251/300, Train metric: 6.0789
Estimator 252/300, Train metric: 6.0865
Estimator 253/300, Train metric: 6.0385
Estimator 254/300, Train metric: 6.0455
Estimator 255/300, Train metric: 6.0139
Estimator 256/300, Train metric: 5.9809
Estimator 257/300, Train metric: 5.9833
Estimator 258/300, Train metric: 5.9267


Estimator 270/300, Train metric: 5.6924
Estimator 271/300, Train metric: 5.6954
Estimator 272/300, Train metric: 5.6684
Estimator 273/300, Train metric: 5.6512
Estimator 274/300, Train metric: 5.6027
Estimator 275/300, Train metric: 5.5584
Estimator 276/300, Train metric: 5.5620
Estimator 277/300, Train metric: 5.5652
Estimator 278/300, Train metric: 5.5403
Estimator 279/300, Train metric: 5.5374
Estimator 280/300, Train metric: 5.5418
Estimator 281/300, Train metric: 5.5485
Estimator 282/300, Train metric: 5.5233
Estimator 283/300, Train metric: 5.5002
Estimator 284/300, Train metric: 5.5035
Estimator 285/300, Train metric: 5.4777
Estimator 286/300, Train metric: 5.4418
Estimator 287/300, Train metric: 5.3929
Estimator 288/300, Train metric: 5.3943
Estimator 289/300, Train metric: 5.3629
Estimator 290/300, Train metric: 5.3620
Estimator 291/300, Train metric: 5.3140
Estimator 292/300, Train metric: 5.3226
Estimator 293/300, Train metric: 5.3074
Estimator 294/300, Train metric: 5.2770


In [16]:
plot_best_scores(best_scores_autosampler,"./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/hyperparameter_tuning/test_results.xlsx")

In [17]:
# FIX B10 (non-math): `model_classes` was used by generate_interpretml_explanations_summary_pruners but never
# defined in this notebook (NameError in the last cell, also in v1.2). Same mapping as in the other notebooks.
model_classes = {
    'Random Forest': RandomForestRegressor,
    'Gradient Boosting': GradientBoostingRegressor,
    'XGBoost': XGBRegressor,
    'LightGBM': LGBMRegressor,
    'GPBoost': GPBoostRegressor,
    'CatBoost': CatBoostRegressor,
    'HistGradientBoosting': HistGradientBoostingRegressor,
    'TabNet': TabNetRegressor,
    'NGBoost': NGBRegressor
}
generate_interpretml_explanations_summary_pruners(best_scores_autosampler, X_train, y_train, x_test, feature_names,excel_file_path = "./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/hyperparameter_tuning/test_results.xlsx")

  0%|          | 0/47 [00:00<?, ?it/s]

  0%|          | 0/47 [00:00<?, ?it/s]

  0%|          | 0/47 [00:00<?, ?it/s]

  0%|          | 0/47 [00:00<?, ?it/s]

  0%|          | 0/47 [00:00<?, ?it/s]

  0%|          | 0/47 [00:00<?, ?it/s]

... (12 outputs left out)

epoch 34 | loss: 261.24942| val_0_mse: 936.2865|  0:00:02s
epoch 35 | loss: 235.55675| val_0_mse: 805.74359|  0:00:02s
epoch 36 | loss: 206.17789| val_0_mse: 889.69763|  0:00:02s
epoch 37 | loss: 253.97694| val_0_mse: 797.61853|  0:00:02s

Early stopping occurred at epoch 37 with best_epoch = 27 and best_val_0_mse = 648.30652


  0%|          | 0/47 [00:00<?, ?it/s]

  0%|          | 0/47 [00:00<?, ?it/s]

Model PGBM is not supported or not available.
